# Does graph-NCA pre-training accelerate a small language model?

This notebook executes **historical matched-total-budget pilot 1**.
Its first returned GPU run did not demonstrate a graph benefit. Scratch receives
1000 text updates while synthetic arms receive 100 synthetic + 900 text updates.
It is not the corrected equal-language-stage transfer protocol. Read the embedded
methodology audit before interpreting results. A positive result is not guaranteed.
It contains all source code; no repository checkout is required.

**Before Run All:** select one available GPU accelerator and enable Internet for
the public Tiny Shakespeare download. Two T4s, if allocated, are not both used.
Keep the notebook private unless you choose to publish it. Save a version with
outputs to preserve the checkpoints. No paid API or rental is used.

The pilot trains the same 1,857,024-parameter causal byte transformer across five
arms and three paired seeds: target text only, separate text, grid NCA, graph NCA,
and shuffled graph trajectories. All arms receive 4,096,000 training tokens per
seed. The declared target is 2.5 validation nats/byte; unreached targets remain
unresolved. Pipeline timing includes synthetic generation and pre-training.
This is a binary graph-topology pilot, not a reproduction of the larger-scale
[2026 NCA paper](https://arxiv.org/abs/2603.10055).

Runtime must be measured on the GPU you receive. Deterministic fp32 favors
auditability over peak throughput. The final archive includes model checkpoints,
metrics, dataset checksums, source, environment, and verification results.


In [ ]:
import os
from pathlib import Path
import sys
import subprocess
from datetime import datetime, timezone

# Set before any CUDA operation, including imports in subprocesses.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path('/kaggle/working/gnn_nca_pilot')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
Path('configs').mkdir(exist_ok=True)
Path('tests').mkdir(exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
print('Project:', ROOT, 'Run:', RUN_ID)

import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle notebook settings, then rerun.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
print('One GPU is used. No distributed training is required.')


## Embedded, reviewable experiment sources
These cells write exactly the project files tested locally.

In [ ]:
%%writefile experiment.py
"""A falsifiable, matched-token pilot of graph-NCA transfer into a byte LM."""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import random
import statistics
import sys
import time
import urllib.request

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

ARMS = ("scratch", "reserve_text", "grid_nca", "graph_nca", "shuffled_graph")
VOCAB = 274  # bytes 0..255, binary 2x2 patches 256..271, frame markers 272..273
TEXT_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def dump(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def sync(device):
    if str(device).startswith("cuda"):
        torch.cuda.synchronize(device)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_config(path):
    c = json.loads(Path(path).read_text(encoding="utf-8"))
    assert c["width"] % c["heads"] == 0
    assert 0 < c["pre_steps"] < c["total_steps"]
    assert c["grid"] % 2 == 0 and c["grid"] >= 4
    assert c["frames"] * ((c["grid"] // 2) ** 2 + 2) > c["context"]
    assert c["precision"] == "fp32", "Pilot uses consistent fp32 on CPU and GPU."
    assert len(set(c["seeds"])) == len(c["seeds"]) and c["seeds"]
    return c


def neighbors(side, rng=None, fraction=0.0):
    """Eight typed incoming edges; periodic Moore lattice or degree-matched rewiring.

    Each relation is a permutation: rewiring preserves incoming AND outgoing degree,
    with no self or duplicate neighbors. Slot labels retain lattice direction labels.
    """
    ids = np.arange(side * side).reshape(side, side)
    offsets = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]
    table = np.stack([np.roll(ids, (-dy, -dx), (0, 1)).ravel() for dy, dx in offsets], 1)
    if fraction:
        assert rng is not None
        for r in range(8):
            for _ in range(round(side * side * fraction / 2)):
                for _attempt in range(100):
                    a, b = rng.choice(side * side, 2, replace=False)
                    va, vb = table[a, r], table[b, r]
                    if vb != a and va != b and vb not in table[a] and va not in table[b]:
                        table[a, r], table[b, r] = vb, va
                        break
    return table


def graph_step(x, edges, w1, b1, w2, b2):
    # x [batch,nodes]; shared relational message function, then a shared node MLP.
    # One independently sampled frozen neural rule per trajectory (batch member).
    local = torch.cat((x[..., None], x[:, edges]), -1).float() * 2 - 1
    hidden = F.relu(torch.einsum("bnr,brh->bnh", local, w1) + b1[:, None])
    return (torch.einsum("bnh,bhk->bnk", hidden, w2) + b2[:, None]).argmax(-1)


def patch_tokens(frames, side):
    # Spatial patches use a fixed node order even on a graph. This serialization
    # favors the lattice and is a deliberate, documented pilot limitation.
    x = frames.reshape(*frames.shape[:-1], side, side)
    p = x[..., 0::2, 0::2] + 2*x[..., 0::2, 1::2] + 4*x[..., 1::2, 0::2] + 8*x[..., 1::2, 1::2]
    p = p.flatten(-2) + 256
    start = torch.full((*p.shape[:-1], 1), 272, device=p.device)
    end = torch.full_like(start, 273)
    return torch.cat((start, p, end), -1).flatten(1)


@torch.no_grad()
def generate(c, topology, device):
    seed_all(c["data_seed"])
    rng = np.random.default_rng(c["data_seed"])
    edge_array = neighbors(c["grid"], rng, c["rewire_fraction"] if topology == "graph" else 0)
    edges = torch.as_tensor(edge_array, dtype=torch.long, device=device)
    chunks, changed, fixed, ratios = [], [], [], []
    sync(device)
    start = time.perf_counter()
    for offset in range(0, c["trajectories"], 64):
        b = min(64, c["trajectories"] - offset)
        h = c["rule_hidden"]
        x = torch.randint(2, (b, c["grid"]**2), device=device)
        w1 = torch.randn(b, 9, h, device=device) / 3
        b1 = torch.randn(b, h, device=device) * 0.2
        w2 = torch.randn(b, h, 2, device=device) / math.sqrt(h)
        b2 = torch.randn(b, 2, device=device) * 0.2
        frames = [x]
        for _ in range(c["frames"] - 1):
            y = graph_step(x, edges, w1, b1, w2, b2)
            changed.append(float((x != y).float().mean()))
            frames.append(y)
            x = y
        fixed.extend((frames[-1] == frames[-2]).all(-1).cpu().tolist())
        chunk = patch_tokens(torch.stack(frames, 1), c["grid"]).cpu().numpy().astype(np.uint16)
        chunks.append(chunk)
        for row in chunk:
            buf = row.astype("<u2").tobytes()
            ratios.append(len(gzip.compress(buf, mtime=0))/len(buf))
    sync(device)
    tokens = np.concatenate(chunks)
    raw = tokens.astype("<u2").tobytes()
    stats = {"generation_seconds": time.perf_counter()-start,
             "tokens": tokens.size, "gzip_ratio": len(gzip.compress(raw, mtime=0))/len(raw),
             "mean_cell_change": statistics.mean(changed),
             "final_fixed_point_fraction": statistics.mean(fixed),
             "per_trajectory_gzip_quartiles": np.quantile(ratios, [.25, .5, .75]).tolist(),
             "distinct_patch_tokens": int(np.unique(tokens[(tokens >= 256) & (tokens < 272)]).size),
             "changed_edges_fraction": float(np.mean(edge_array != neighbors(c["grid"]))),
             "topology_sha256": hashlib.sha256(edge_array.tobytes()).hexdigest()}
    return tokens, stats, edge_array


def shuffle_trajectories(tokens, seed):
    """Destroy patch dynamics, preserving every row's unigram counts AND markers."""
    rng = np.random.default_rng(seed)
    out = tokens.copy()
    for row in out:
        mask = row < 272
        row[mask] = rng.permutation(row[mask])
    return out


def prepare(args, c, device):
    root = Path(args.data)
    root.mkdir(parents=True, exist_ok=True)
    # Fail closed: cached artifacts must match config and checksums at run time.
    source = Path(args.text) if args.text else root / "input.txt"
    if not source.exists():
        print("Downloading Tiny Shakespeare...", flush=True)
        urllib.request.urlretrieve(TEXT_URL, source)
    raw = source.read_bytes()
    text_start = time.perf_counter()
    n = len(raw)
    cuts = [0, int(.2*n), int(.8*n), int(.9*n), n]
    manifest = {"config": c, "source": str(source.resolve()),
                "source_url": TEXT_URL if not args.text else None,
                "source_sha256": digest(source), "splits": {}, "synthetic": {}, "files": {}}
    for i, name in enumerate(("reserve", "train", "val", "test")):
        a, b = cuts[i:i+2]
        if b-a < (c["context"]+1)*c["batch"]:
            raise ValueError("Text split too short for this configuration")
        np.save(root / f"{name}.npy", np.frombuffer(raw[a:b], dtype=np.uint8).astype(np.uint16))
        manifest["splits"][name] = {"start_byte": a, "end_byte": b, "tokens": b-a}
    manifest["text_preprocessing_seconds"] = time.perf_counter()-text_start
    for kind in ("grid", "graph"):
        tokens, stats, edges = generate(c, kind, device)
        t = time.perf_counter()
        np.save(root / f"{kind}.npy", tokens)
        np.save(root / f"{kind}_edges.npy", edges)
        stats["generation_seconds"] += time.perf_counter()-t
        manifest["synthetic"][kind] = stats
        if kind == "graph":
            t = time.perf_counter()
            shuffled = shuffle_trajectories(tokens, c["data_seed"]+1)
            np.save(root / "shuffled.npy", shuffled)
            manifest["synthetic"]["shuffled"] = {
                "generation_seconds": stats["generation_seconds"] + time.perf_counter()-t,
                "parent": "graph", "preserves": "per-trajectory patch histogram and marker positions"}
    for f in sorted(root.glob("*.npy")):
        manifest["files"][f.name] = digest(f)
    dump(root / "manifest.json", manifest)
    print(json.dumps(manifest["synthetic"], indent=2), flush=True)


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c["width"]
        self.heads = c["heads"]
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.proj = nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.norm1(x)).reshape(b, t, 3, self.heads, d//self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(a.transpose(1, 2).reshape(b, t, d))
        return x + self.mlp(self.norm2(x))


class LM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.token = nn.Embedding(VOCAB, c["width"])
        self.position = nn.Embedding(c["context"], c["width"])
        self.blocks = nn.Sequential(*[Block(c) for _ in range(c["layers"])])
        self.norm = nn.LayerNorm(c["width"])
        self.head = nn.Linear(c["width"], VOCAB, bias=False)
        self.head.weight = self.token.weight
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Embedding, nn.Linear)):
            nn.init.normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        z = self.token(x) + self.position(torch.arange(x.shape[1], device=x.device))
        return self.head(self.norm(self.blocks(z)))


def sample_batch(array, c, rng, device):
    t, b = c["context"], c["batch"]
    if array.ndim == 1:
        starts = rng.integers(0, len(array)-t, size=b)
        chunk = array[starts[:, None] + np.arange(t+1)[None, :]]
    else:
        rows = rng.integers(0, array.shape[0], size=b)
        starts = rng.integers(0, array.shape[1]-t, size=b)
        chunk = array[rows[:, None], starts[:, None]+np.arange(t+1)[None, :]]
    chunk = torch.as_tensor(chunk.astype(np.int64), device=device)
    return chunk[:, :-1], chunk[:, 1:]


def eval_batches(array, c, device):
    # Non-overlapping targets; every arm uses exactly the same fixed subset.
    t, b = c["context"], c["batch"]
    count = min(c["eval_batches"]*b, (len(array)-1)//t)
    blocks = np.stack([array[i*t:i*t+t+1] for i in range(count)])
    return [torch.as_tensor(blocks[i:i+b].astype(np.int64), device=device) for i in range(0, count, b)]


@torch.no_grad()
def evaluate(model, batches):
    model.eval()
    total, n = 0., 0
    for z in batches:
        logits = model(z[:, :-1])
        total += float(F.cross_entropy(logits.reshape(-1, VOCAB), z[:, 1:].reshape(-1), reduction="sum"))
        n += z[:, 1:].numel()
    model.train()
    return total/n


def lr_at(step, length, c):
    warm = min(c["warmup"], max(1, length//10))
    if step < warm:
        return c["lr"]*(step+1)/warm
    phase = (step-warm)/max(1, length-warm-1)
    return c["lr"]*(.1 + .9*(1+math.cos(math.pi*phase))/2)


def warmup_training(c, array, device):
    """Pay lazy optimizer/kernel startup once, before any measured arm.

    The temporary model is discarded. Each real arm reseeds and initializes from
    scratch. The shared warmup cost is conservatively charged to every arm.
    """
    seed_all(0)
    model = LM(c).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=c["lr"])
    rng = np.random.default_rng(0)
    for _ in range(3):
        x, y = sample_batch(array, c, rng, device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1)).backward()
        opt.step()
    sync(device)


def load_data(root, c):
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    if manifest["config"] != c:
        raise ValueError("Dataset/config mismatch; prepare a new data directory")
    for name, expected in manifest["files"].items():
        if digest(root / name) != expected:
            raise ValueError(f"Checksum mismatch: {name}")
    names = ("reserve", "train", "val", "test", "grid", "graph", "shuffled")
    return {n: np.load(root / f"{n}.npy") for n in names}, manifest


def run_one(c, arm, seed, arrays, manifest, device, out):
    sync(device)
    start = time.perf_counter()
    seed_all(seed)
    model = LM(c).to(device)
    initial_hash = hashlib.sha256(b"".join(p.detach().cpu().numpy().tobytes() for p in model.parameters())).hexdigest()
    parameter_count = sum(p.numel() for p in model.parameters())
    # Synthetic symbols have no byte semantics. Reset synthetic embeddings only;
    # text baselines retain theirs. Graph/grid/shuffled use identical resets.
    embedding_init = model.token.weight.detach().clone()
    pre_name = {"scratch": "train", "reserve_text": "reserve", "grid_nca": "grid",
                "graph_nca": "graph", "shuffled_graph": "shuffled"}[arm]
    prep_seconds = (manifest["synthetic"].get(pre_name, {}).get("generation_seconds", 0.)
                    + manifest["text_preprocessing_seconds"] + manifest.get("load_seconds", 0.))
    validation = eval_batches(arrays["val"], c, device)
    pre_rng = np.random.default_rng(seed+100)
    target_rng = np.random.default_rng(seed+200)
    train_seconds, rows, target_step = 0., [], 0
    out = Path(out)
    out.mkdir(parents=True, exist_ok=False)

    def record(step, stage):
        loss = evaluate(model, validation)
        sync(device)
        pre_done = min(step, c["pre_steps"])
        # scratch's first-stage text tokens count as downstream target exposure.
        target_tokens = (max(0, step-c["pre_steps"])+(pre_done if arm == "scratch" else 0))*c["batch"]*c["context"]
        row = {"arm": arm, "seed": seed, "step": step, "stage": stage,
               "total_tokens": step*c["batch"]*c["context"], "target_text_tokens": target_tokens,
               "val_nll": loss, "val_perplexity": math.exp(loss),
               "train_seconds": train_seconds,
               "pipeline_seconds": prep_seconds+time.perf_counter()-start}
        rows.append(row)
        with (out / "metrics.jsonl").open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, allow_nan=False)+"\n")
        print(f"{arm} seed={seed} step={step} val={loss:.4f} total_tokens={row['total_tokens']}", flush=True)

    record(0, "initial")
    for stage, length, name, rng in (("pre", c["pre_steps"], pre_name, pre_rng),
                                     ("target", c["total_steps"]-c["pre_steps"], "train", target_rng)):
        if stage == "target" and arm in ("grid_nca", "graph_nca", "shuffled_graph"):
            with torch.no_grad():
                model.token.weight.copy_(embedding_init)
        # Every arm resets Adam and stage scheduler at the same boundary.
        optimizer = torch.optim.AdamW(model.parameters(), lr=c["lr"], weight_decay=c["weight_decay"])
        if stage == "target":
            record(c["pre_steps"], "target_start")
        for local_step in range(length):
            sync(device)
            tick = time.perf_counter()
            x, y = sample_batch(arrays[name], c, rng, device)
            for group in optimizer.param_groups:
                group["lr"] = lr_at(local_step, length, c)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()
            sync(device)
            train_seconds += time.perf_counter()-tick
            target_step += 1
            if target_step % c["eval_every"] == 0 or local_step == length-1:
                record(target_step, stage)
    test_nll = evaluate(model, eval_batches(arrays["test"], c, device))
    torch.save({"model": model.state_dict(), "config": c, "arm": arm, "seed": seed,
                "initial_sha256": initial_hash, "manifest": manifest}, out / "model.pt")
    summary = {"arm": arm, "seed": seed, "parameters": parameter_count,
               "initial_sha256": initial_hash, "test_nll": test_nll,
               "test_perplexity": math.exp(test_nll), "final_val_nll": rows[-1]["val_nll"],
               "train_seconds": train_seconds, "preparation_seconds": prep_seconds,
               "val_evaluated_tokens": sum(z[:, 1:].numel() for z in validation),
               "embedding_transfer": "reset" if arm in ("grid_nca", "graph_nca", "shuffled_graph") else "preserved",
               "total_tokens": rows[-1]["total_tokens"],
               "approx_train_flops_6PD": 6*parameter_count*rows[-1]["total_tokens"],
               "model_sha256": digest(out / "model.pt")}
    dump(out / "summary.json", summary)
    return summary


def first_hit(rows, threshold, key):
    # Text baselines can reach the target during their first stage. Synthetic
    # stages are excluded until their byte embeddings have been restored.
    eligible = [r for r in rows if (r["stage"] in ("target_start", "target") or
                (r.get("arm") in ("scratch", "reserve_text") and r["stage"] == "pre"))
                and r["val_nll"] <= threshold]
    return min((r[key] for r in eligible), default=None)


def report(out, c):
    out = Path(out)
    summaries, curves = {}, {}
    for arm in ARMS:
        for seed in c["seeds"]:
            folder = out / f"{arm}_{seed}"
            if (folder / "summary.json").exists():
                summaries[arm, seed] = json.loads((folder / "summary.json").read_text())
                curves[arm, seed] = [json.loads(x) for x in (folder / "metrics.jsonl").read_text().splitlines()]
    comparisons = []
    for baseline in ("scratch", "grid_nca", "reserve_text", "shuffled_graph"):
        for seed in c["seeds"]:
            if (baseline, seed) not in summaries or ("graph_nca", seed) not in summaries:
                continue
            a, b = summaries[baseline, seed], summaries["graph_nca", seed]
            row = {"baseline": baseline, "seed": seed,
                   "graph_minus_baseline_test_nll": b["test_nll"]-a["test_nll"]}
            for key in ("total_tokens", "target_text_tokens", "train_seconds", "pipeline_seconds"):
                ah = first_hit(curves[baseline, seed], c["target_nll"], key)
                bh = first_hit(curves["graph_nca", seed], c["target_nll"], key)
                row[f"{key}_speedup"] = ah/bh if ah is not None and bh is not None and bh > 0 else None
            row["target_status"] = "both_reached" if row["pipeline_seconds_speedup"] is not None else "unresolved_not_both_reached"
            comparisons.append(row)
    aggregate = []
    for baseline in ("scratch", "grid_nca", "reserve_text", "shuffled_graph"):
        pairs = [r for r in comparisons if r["baseline"] == baseline]
        if not pairs:
            continue
        values = [r["graph_minus_baseline_test_nll"] for r in pairs]
        rng = np.random.default_rng(123)
        boot = np.mean(rng.choice(values, (10000, len(values)), replace=True), 1)
        speedups = [r["pipeline_seconds_speedup"] for r in pairs if r["pipeline_seconds_speedup"] is not None]
        speedup_ci = None
        if len(speedups) >= 2:
            samples = np.exp(np.mean(rng.choice(np.log(speedups), (10000, len(speedups)), replace=True), 1))
            speedup_ci = np.quantile(samples, [.025, .975]).tolist()
        aggregate.append({"baseline": baseline, "paired_seeds": len(values),
                          "mean_test_nll_delta": statistics.mean(values),
                          "bootstrap_95_ci_exploratory": np.quantile(boot, [.025, .975]).tolist() if len(values) >= 2 else None,
                          "pairs_both_reached_target": len(speedups),
                          "geomean_pipeline_speedup_reached_pairs_only": math.exp(statistics.mean(map(math.log, speedups))) if speedups else None,
                          "speedup_bootstrap_95_ci_exploratory": speedup_ci,
                          "speedup_caution": "Exclude no seed silently: if any seed is censored, reached-only speedup does not establish a suite-wide gain.",
                          "caution": "Three seeds are exploratory; one seed has no uncertainty estimate."})
    dump(out / "report.json", {"threshold_nll": c["target_nll"], "comparisons": comparisons,
                               "aggregate": aggregate, "claim": "No automatic proof: inspect paired results and measured time-to-target."})
    if summaries:
        with (out / "summary.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(next(iter(summaries.values()))))
            writer.writeheader()
            writer.writerows(summaries.values())
    write_plot(out, curves)
    print(json.dumps(aggregate, indent=2), flush=True)


def write_plot(out, curves):
    # Portable SVG: no plotting dependency needed in Kaggle or on a CPU laptop.
    colors = dict(zip(ARMS, ("#64748b", "#b7791f", "#2563eb", "#059669", "#db2777")))
    rows = [r for curve in curves.values() for r in curve if r["stage"] in ("target_start", "target")]
    if not rows:
        return
    lo, hi = min(r["val_nll"] for r in rows)-.05, max(r["val_nll"] for r in rows)+.05
    xmax = max(r["total_tokens"] for r in rows)
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520" viewBox="0 0 900 520">',
             '<rect width="900" height="520" fill="white"/>',
             '<text x="70" y="28" font-family="sans-serif" font-size="20">Validation loss vs total training tokens (all stages counted)</text>',
             '<path d="M70 50V440H850" fill="none" stroke="#334155"/>']
    for i in range(5):
        y = 440-i*95
        value = lo+(hi-lo)*i/4
        parts.append(f'<text x="12" y="{y+4}" font-family="sans-serif" font-size="12">{value:.2f}</text>')
        parts.append(f'<text x="{70+i*195}" y="462" font-family="sans-serif" font-size="12">{xmax*i/4:,.0f}</text>')
    for (arm, seed), curve in curves.items():
        points = " ".join(f'{70+780*r["total_tokens"]/xmax:.1f},{440-380*(r["val_nll"]-lo)/(hi-lo):.1f}'
                          for r in curve if r["stage"] in ("target_start", "target"))
        parts.append(f'<polyline points="{points}" fill="none" stroke="{colors[arm]}" stroke-width="1.8" opacity="0.65"><title>{arm}, seed {seed}</title></polyline>')
    for i, arm in enumerate(ARMS):
        parts.append(f'<text x="{70+i*160}" y="500" fill="{colors[arm]}" font-family="sans-serif" font-size="12">{arm}</text>')
    parts.append('</svg>')
    (out / "curves.svg").write_text("\n".join(parts), encoding="utf-8")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command", choices=("prepare", "run", "report", "all"))
    parser.add_argument("--config", default="configs/pilot.json")
    parser.add_argument("--data", default="data/pilot")
    parser.add_argument("--out", default="runs/pilot")
    parser.add_argument("--text", help="Existing UTF-8/plain text corpus; no download needed")
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--arms", nargs="+", choices=ARMS, default=list(ARMS))
    args = parser.parse_args()
    c = load_config(args.config)
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    # Determinism is checked, not silently best-effort; GPU CUBLAS config must be
    # set before launching Python, as done by the notebook.
    torch.use_deterministic_algorithms(True)
    if args.command in ("prepare", "all"):
        prepare(args, c, device)
    if args.command in ("run", "all"):
        load_start = time.perf_counter()
        arrays, manifest = load_data(args.data, c)
        warmup_training(c, arrays["train"], device)
        manifest["load_seconds"] = time.perf_counter()-load_start
        root = Path(args.out)
        root.mkdir(parents=True, exist_ok=True)
        env_path = root / "environment.json"
        if env_path.exists():
            raise FileExistsError("Output directory already has a run. Choose a new --out path.")
        dump(env_path, {"config": c, "python": sys.version, "torch": torch.__version__,
                        "numpy": np.__version__, "platform": platform.platform(), "device": device,
                        "gpu": torch.cuda.get_device_name() if device == "cuda" else None,
                        "cuda": torch.version.cuda, "source_sha256": digest(__file__),
                        "tf32": torch.backends.cuda.matmul.allow_tf32,
                        "data_manifest_sha256": digest(Path(args.data)/"manifest.json"),
                        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
                        "interpretation": "CPU/smoke outputs validate execution only; not a GPU speed claim."})
        # Counterbalance arm order by seed to reduce systematic warm-up/order bias.
        for i, seed in enumerate(c["seeds"]):
            order = args.arms[i % len(args.arms):]+args.arms[:i % len(args.arms)]
            for arm in order:
                run_one(c, arm, seed, arrays, manifest, device, root / f"{arm}_{seed}")
    if args.command in ("run", "report", "all"):
        report(args.out, c)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile verify.py
"""Re-load saved checkpoints and independently reproduce held-out test loss."""
import argparse
import json
from pathlib import Path

import torch

import experiment as e


def verify(root, data_root, device="cpu"):
    root = Path(root)
    env = json.loads((root / "environment.json").read_text())
    c = env["config"]
    data, _ = e.load_data(data_root, c)
    tests = e.eval_batches(data["test"], c, device)
    seen = {}
    records = []
    for folder in sorted(root.iterdir()):
        if not (folder / "summary.json").exists():
            continue
        summary = json.loads((folder / "summary.json").read_text())
        assert e.digest(folder / "model.pt") == summary["model_sha256"], "Checkpoint checksum mismatch"
        saved = torch.load(folder / "model.pt", map_location=device, weights_only=True)
        assert saved["config"] == c
        assert saved["arm"] == summary["arm"] and saved["seed"] == summary["seed"]
        assert saved["initial_sha256"] == summary["initial_sha256"]
        assert summary["total_tokens"] == c["total_steps"]*c["batch"]*c["context"]
        if summary["seed"] in seen:
            assert summary["initial_sha256"] == seen[summary["seed"]], "Unpaired initialization"
        seen[summary["seed"]] = summary["initial_sha256"]
        model = e.LM(c).to(device)
        model.load_state_dict(saved["model"], strict=True)
        loss = e.evaluate(model, tests)
        delta = abs(loss-summary["test_nll"])
        if delta > 1e-4:
            raise AssertionError(f"Test loss mismatch: {folder.name}, delta={delta}")
        records.append({"arm": saved["arm"], "seed": saved["seed"],
                        "reproduced_test_nll": loss, "absolute_error": delta,
                        "checkpoint_sha256_valid": True})
    if not records:
        raise ValueError("No completed checkpoints found")
    result = {"device": device, "checkpoints_verified": len(records), "records": records,
              "limitation": "Verification establishes artifact consistency, not a scientific speedup."}
    e.dump(root / "verification.json", result)
    print(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--out", default="runs/pilot")
    p.add_argument("--data", default="data/pilot")
    p.add_argument("--device", choices=("cpu", "cuda"), default="cpu")
    a = p.parse_args()
    torch.set_num_threads(4)
    verify(a.out, a.data, a.device)


In [ ]:
%%writefile plot_results.py
"""Export standard Matplotlib figures from recorded measurements."""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
from matplotlib import pyplot as plt

from experiment import ARMS


def plot(root):
    root = Path(root)
    env = json.loads((root / "environment.json").read_text())
    colors = dict(zip(ARMS, ("#64748b", "#b7791f", "#2563eb", "#059669", "#db2777")))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for arm in ARMS:
        for index, seed in enumerate(env["config"]["seeds"]):
            path = root / f"{arm}_{seed}" / "metrics.jsonl"
            if not path.exists():
                continue
            rows = [json.loads(line) for line in path.read_text().splitlines()]
            # Eligible byte-LM evaluations; no synthetic-symbol pre-stage losses.
            rows = [r for r in rows if r["stage"] in ("target_start", "target") or
                    (arm in ("scratch", "reserve_text") and r["stage"] == "pre")]
            for ax, key, scale in zip(axes, ("total_tokens", "pipeline_seconds"), (1e6, 1)):
                ax.plot([r[key]/scale for r in rows], [r["val_nll"] for r in rows],
                        color=colors[arm], alpha=.7, linewidth=1.5,
                        label=arm if index == 0 else None)
    for ax in axes:
        ax.axhline(env["config"]["target_nll"], color="#111827", linestyle="--", linewidth=.8, label="Declared target")
        ax.set_ylabel("Validation NLL (nats per byte)")
        ax.grid(alpha=.2)
    axes[0].set_xlabel("Total training tokens (millions; both stages)")
    axes[1].set_xlabel("Pipeline time (seconds; preparation included)")
    axes[0].legend(fontsize=8)
    device = env["device"].upper()
    fig.suptitle(f"Graph-NCA transfer — {device}, {len(env['config']['seeds'])} paired seed(s)")
    fig.savefig(root / "curves.svg")
    fig.savefig(root / "curves.png", dpi=180)
    plt.close(fig)
    print(f"Exported {root / 'curves.svg'} and curves.png")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--out", default="runs/pilot")
    plot(p.parse_args().out)


In [ ]:
%%writefile tests/test_experiment.py
"""Scientific validity checks. Run: python -m unittest discover -s tests -v."""
import unittest

import numpy as np
import torch
from torch.nn import functional as F

import experiment as e


class ScientificChecks(unittest.TestCase):
    def setUp(self):
        self.c = e.load_config("configs/smoke.json")
        e.seed_all(123)

    def test_degree_matched_graph_without_self_or_duplicate_edges(self):
        n = 64
        lattice = e.neighbors(8)
        graph = e.neighbors(8, np.random.default_rng(10), .25)
        self.assertGreater(np.mean(graph != lattice), .05)
        for relation in graph.T:
            np.testing.assert_array_equal(np.sort(relation), np.arange(n))
        for node, row in enumerate(graph):
            self.assertEqual(len(set(row)), 8)
            self.assertNotIn(node, row)
        np.testing.assert_array_equal(graph, e.neighbors(8, np.random.default_rng(10), .25))

    def test_lattice_message_passing_equals_convolution(self):
        # Proves the grid reference IS an NCA, and that writing it as a GNN
        # alone does not constitute a different scientific treatment.
        side, hidden = 8, 16
        x = torch.randint(2, (1, side*side))
        w1, b1 = torch.randn(1, 9, hidden), torch.randn(1, hidden)
        w2, b2 = torch.randn(1, hidden, 2), torch.randn(1, 2)
        edges = torch.tensor(e.neighbors(side))
        graph = e.graph_step(x, edges, w1, b1, w2, b2)
        # graph input order is self, then the 8 row-major offsets.
        slots = [1, 2, 3, 4, 0, 5, 6, 7, 8]
        kernel = w1[0, slots].T.reshape(hidden, 1, 3, 3)
        field = (x.float()*2-1).reshape(1, 1, side, side)
        z = F.relu(F.conv2d(F.pad(field, (1, 1, 1, 1), mode="circular"), kernel, b1[0]))
        logits = F.conv2d(z, w2[0].T.reshape(2, hidden, 1, 1), b2[0])
        conv = logits.argmax(1).reshape(1, -1)
        torch.testing.assert_close(graph, conv)

    def test_binary_patch_vocabulary_is_bijective(self):
        states = torch.tensor([[(i >> j) & 1 for j in range(4)] for i in range(16)])[:, None, :]
        encoded = e.patch_tokens(states, 2)
        torch.testing.assert_close(encoded[:, 1], torch.arange(256, 272))
        self.assertTrue(torch.all(encoded[:, 0] == 272))
        self.assertTrue(torch.all(encoded[:, 2] == 273))

    def test_shuffle_preserves_counts_and_boundaries_but_changes_order(self):
        a = np.tile(np.r_[272, np.arange(256, 272), 273], (8, 4)).astype(np.uint16)
        b = e.shuffle_trajectories(a, 42)
        np.testing.assert_array_equal(np.sort(a, axis=1), np.sort(b, axis=1))
        np.testing.assert_array_equal(a >= 272, b >= 272)
        self.assertFalse(np.array_equal(a, b))

    def test_batches_never_cross_trajectory_boundaries(self):
        c = dict(self.c, context=8, batch=64)
        a = np.stack([np.full(40, i) for i in range(5)])
        x, y = e.sample_batch(a, c, np.random.default_rng(1), "cpu")
        self.assertTrue(torch.all(x == x[:, :1]))
        self.assertTrue(torch.all(y == x))

    def test_causal_attention_cannot_see_future_tokens(self):
        m = e.LM(self.c).eval()
        x = torch.randint(256, (2, 16))
        y = x.clone()
        y[:, 8:] = torch.randint(256, (2, 8))
        torch.testing.assert_close(m(x)[:, :8], m(y)[:, :8], rtol=0, atol=1e-6)

    def test_identical_initialization_and_gradient_update(self):
        e.seed_all(17)
        a = e.LM(self.c)
        e.seed_all(17)
        b = e.LM(self.c)
        for x, y in zip(a.parameters(), b.parameters()):
            torch.testing.assert_close(x, y, rtol=0, atol=0)
        before = a.blocks[0].qkv.weight.detach().clone()
        opt = torch.optim.AdamW(a.parameters(), lr=.01)
        logits = a(torch.randint(256, (2, 16)))
        F.cross_entropy(logits.reshape(-1, e.VOCAB), torch.randint(256, (32,))).backward()
        opt.step()
        self.assertFalse(torch.equal(before, a.blocks[0].qkv.weight))

    def test_embedding_reset_preserves_backbone_and_weight_tying(self):
        m = e.LM(self.c)
        emb = m.token.weight.detach().clone()
        with torch.no_grad():
            m.token.weight.add_(1)
            m.blocks[0].qkv.weight.add_(1)
        backbone = m.blocks[0].qkv.weight.detach().clone()
        with torch.no_grad():
            m.token.weight.copy_(emb)
        torch.testing.assert_close(m.blocks[0].qkv.weight, backbone)
        self.assertIs(m.head.weight, m.token.weight)

    def test_unreached_target_is_censored(self):
        rows = [{"stage": "pre", "val_nll": 1., "total_tokens": 10},
                {"stage": "target_start", "val_nll": 3., "total_tokens": 20},
                {"stage": "target", "val_nll": 2., "total_tokens": 30}]
        self.assertIsNone(e.first_hit(rows, 1.5, "total_tokens"))
        self.assertEqual(e.first_hit(rows, 2.5, "total_tokens"), 30)

    def test_text_baseline_can_reach_target_in_first_stage(self):
        rows = [{"arm": "scratch", "stage": "pre", "val_nll": 2., "total_tokens": 10},
                {"arm": "scratch", "stage": "target", "val_nll": 1.5, "total_tokens": 30}]
        self.assertEqual(e.first_hit(rows, 2.5, "total_tokens"), 10)

    def test_generation_reproducible_and_valid(self):
        a, stats, edges = e.generate(self.c, "graph", "cpu")
        b, _, edges2 = e.generate(self.c, "graph", "cpu")
        np.testing.assert_array_equal(a, b)
        np.testing.assert_array_equal(edges, edges2)
        self.assertTrue(np.all((a >= 256) & (a < e.VOCAB)))
        self.assertGreater(stats["mean_cell_change"], 0)
        self.assertGreater(stats["generation_seconds"], 0)


if __name__ == "__main__":
    torch.set_num_threads(2)
    unittest.main()


In [ ]:
%%writefile configs/pilot.json
{
  "seeds": [17, 29, 43],
  "layers": 4,
  "heads": 4,
  "width": 192,
  "context": 128,
  "batch": 32,
  "pre_steps": 100,
  "total_steps": 1000,
  "eval_every": 50,
  "eval_batches": 16,
  "lr": 0.0006,
  "warmup": 20,
  "weight_decay": 0.1,
  "target_nll": 2.5,
  "grid": 8,
  "frames": 16,
  "trajectories": 1024,
  "rule_hidden": 16,
  "rewire_fraction": 0.25,
  "data_seed": 20261017,
  "precision": "fp32"
}


In [ ]:
%%writefile configs/smoke.json
{
  "seeds": [17],
  "layers": 2,
  "heads": 2,
  "width": 32,
  "context": 32,
  "batch": 2,
  "pre_steps": 2,
  "total_steps": 6,
  "eval_every": 2,
  "eval_batches": 2,
  "lr": 0.0006,
  "warmup": 1,
  "weight_decay": 0.1,
  "target_nll": 2.5,
  "grid": 8,
  "frames": 8,
  "trajectories": 16,
  "rule_hidden": 16,
  "rewire_fraction": 0.25,
  "data_seed": 20261017,
  "precision": "fp32"
}


In [ ]:
%%writefile configs/cpu_check.json
{
  "seeds": [17],
  "layers": 4,
  "heads": 4,
  "width": 192,
  "context": 128,
  "batch": 4,
  "pre_steps": 10,
  "total_steps": 60,
  "eval_every": 10,
  "eval_batches": 4,
  "lr": 0.0006,
  "warmup": 5,
  "weight_decay": 0.1,
  "target_nll": 2.5,
  "grid": 8,
  "frames": 16,
  "trajectories": 64,
  "rule_hidden": 16,
  "rewire_fraction": 0.25,
  "data_seed": 20261017,
  "precision": "fp32"
}


In [ ]:
%%writefile requirements.txt
numpy>=1.26,<3
torch>=2.5,<3
matplotlib>=3.8,<4


In [ ]:
%%writefile README.md
# GNNs in NCA

Research code for testing whether graph neural cellular automata (NCA) data can
improve a transformer's initialization **before language pre-training**.
We compare graph, grid and shuffled synthetic trajectories with text controls.
The GNN is a frozen random-rule data generator, not a component inside the LM.

**Status:** the returned **version 2 GPU transfer experiment** completed on a
Tesla T4. Grid-NCA pre-pre-training reached the paired no-PPT endpoint in
1600–1800 language updates instead of 2000 across three seeds. Shuffled NCA
achieved comparable gains, so this does not establish a dynamics-specific benefit.
Recorded cumulative time was about 13% higher for grid NCA at that reference.
See [the v2 results review](docs/results/transfer_v2_run1/REVIEW.md).
The [results ledger](RESULTS.md) collects current measurements, mathematical
verification and historical evidence in one place.
The detailed [research presentation](docs/presentation/GPU_NCA_Research_Detailed_v2.pptx)
and its [slide/source index](docs/presentation/README.md) cover the theory,
proofs, implementation, notebooks, visualizations and reviewed results.
Graph rewiring has not been tested in v2. The historical
`experiment.py` still implements **matched-total-budget pilot 1**.
The first GPU pilot did not demonstrate a graph benefit. Its scratch model saw
1000 text updates; NCA arms saw 100 synthetic + 900 text updates. The language
stage and scheduler confounds prevent interpreting that result as a clean test
of equal-language-stage pre-pre-training transfer.

Start with [research context](RESEARCH_CONTEXT.md), [methodology audit](METHODOLOGY_AUDIT.md),
and [historical GPU evidence](docs/results/pilot1/REVIEW.md). The v2 result is a
small development experiment with one shared synthetic cache. No general acceleration,
reasoning improvement or successful reproduction of the reference paper is claimed.

## Next run: graph versus grid pre-pre-training, version 3

Upload **[kaggle_nca_transfer_v3.ipynb](kaggle_nca_transfer_v3.ipynb)** to Kaggle,
enable one GPU and Internet, then Run All. This next stage tests graph-NCA PPT
against grid-NCA PPT, with shuffled controls for both and a no-PPT control.
All five enter the same 2000-update language stage after 1000 PPT updates
(zero for scratch). Three paired model/data/topology replicates use common
frozen grid/graph rules selected with the same complexity filter.

The transformer and language settings match v2. Save actual common starting
weights; verify transferred backbones, common heads and all stage checkpoints.
Language training starts only if every grid and graph seed passes the synthetic
gate. The official test remains untouched. Based on the returned T4 run, allow
roughly 2.5–3 hours plus variable preparation/export overhead; the notebook
measures a fresh estimate before training.

Completion prints `WORKFLOW FINISHED: complete` or
`WORKFLOW FINISHED: stopped_at_synthetic_gate`, followed by the download link for
`nca_transfer_v3_results_<timestamp>.zip`. Read [the v3 protocol](docs/TRANSFER_V3_PROTOCOL.md).
V3 has local software checks; a new GPU outcome has not been measured.

## Previous run: pre-pre-training transfer, version 2

Upload **[kaggle_nca_transfer_v2.ipynb](kaggle_nca_transfer_v2.ipynb)** to Kaggle,
enable one available GPU and Internet, then Run All to repeat v2. For the next
graph experiment use v3 above; `kaggle_gnn_nca_pilot.ipynb` is historical pilot 1.

Default: 1000 synthetic updates followed by **2000 equal language updates**,
three paired seeds, with zero synthetic updates for scratch. Categorical NCA,
rule filtering, burn-in, a small RoPE/RMSNorm/SwiGLU transformer, and fresh
WikiText/GPT-2 data replace the first pilot's simplified setup. The confirmation
test split stays untouched by default.
The shared backbone has a 4.58M-parameter synthetic model and a 12.31M-parameter
language model because the two vocabularies require different embedding tables.

The notebook measures a timing estimate on your actual GPU before training.
It saves PPT checkpoints, checks unseen-rule prediction against fitted controls,
and starts the language comparison only if the declared synthetic gate passes.
Language-stage acceleration and cumulative training costs are reported separately.

The last cell exports `nca_transfer_v2_results_<timestamp>.zip`. It reports either
`complete` (language benchmark completed) or `stopped_at_synthetic_gate`
(diagnostics completed; no language benchmark). Both archives are useful to
review; neither status automatically means the hypothesis succeeded.

Read [the v2 protocol](docs/TRANSFER_V2_PROTOCOL.md). Local CLI dependencies are
in `requirements-v2.txt`; additional checked versions are in
`requirements-v2-tested.txt`. Offline checks do not require network packages:

```bash
python scripts/run_logged.py --label v2-scientific -- python -m unittest discover -s tests -p test_transfer_v2.py -v
python scripts/run_logged.py --label v2-integration -- python scripts/smoke_transfer_v2.py
```

## Quick start

The separate [mathematical note](docs/theory/README.md) develops the graph-NCA
concept in LaTeX, including proofs, transfer conditions, counterexamples and
worked numerical derivations.
The companion [Lean proof folder](proofs/lean/README.md) contains 44 checked
statements, a [LaTeX coverage map](proofs/lean/PROOF_MAP.md), reproducible proof
sources and retained verification evidence. Its coverage is partial: the full
entropy, spectral and smoothness arguments remain informal.

Python **3.12** is the checked interpreter. A CPU is sufficient for tests and
smoke checks. Install dependencies in a virtual environment from this directory.

Windows PowerShell:

```powershell
py -3.12 -m venv .venv
.venv\Scripts\python.exe -m pip install -r requirements.txt
.venv\Scripts\python.exe scripts/run_logged.py --label unit-tests -- python -m unittest discover -s tests -v
.venv\Scripts\python.exe scripts/run_logged.py --label offline-smoke -- python scripts/smoke_check.py
```

Linux/macOS:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python scripts/run_logged.py --label unit-tests -- python -m unittest discover -s tests -v
python scripts/run_logged.py --label offline-smoke -- python scripts/smoke_check.py
```

`requirements.txt` defines supported dependency ranges. `requirements-tested.txt`
records the direct versions actually checked on Windows CPU. CUDA requires a
suitable PyTorch build; do not infer GPU reproducibility from CPU checks. Do not
use Python `-O`, which disables assertions in historical scientific checks.

The offline smoke check uses generated fixture text in a temporary directory.
It exercises all five arms, cache checks, training, reporting, checkpoint
verification and plot export without a download. Temporary artifacts are removed;
full captured output is retained in `logs/checks.jsonl`. Fixture losses are not
research results. CI is configured for Python 3.12 on Linux and Windows; a CI
configuration is not evidence that those remote jobs have run.

## Run the historical pilot

Use a fresh data directory and output directory. To retain a CPU smoke run with
the default corpus (first preparation downloads Tiny Shakespeare):

```bash
python experiment.py all --config configs/smoke.json --data data/smoke_001 --out runs/smoke_001 --device cpu
python verify.py --data data/smoke_001 --out runs/smoke_001 --device cpu
python plot_results.py --out runs/smoke_001
```

Supply `--text /path/to/corpus.txt` to avoid downloading. The text must be long
enough for every configured split. Only reuse a cache with its original config.
`prepare`, `run` and `report` can be called separately. `prepare` rewrites its
selected cache: always choose a fresh path for changed data/configuration. Failed
runs are not automatically resumable; retain them and choose a new output path.

For the full historical GPU pilot, set deterministic CUDA configuration before
starting Python:

```bash
export CUBLAS_WORKSPACE_CONFIG=:4096:8
python experiment.py all --config configs/pilot.json --data data/pilot_001 --out runs/pilot_001 --device cuda
python verify.py --data data/pilot_001 --out runs/pilot_001 --device cuda
python plot_results.py --out runs/pilot_001
```

In PowerShell use `$env:CUBLAS_WORKSPACE_CONFIG = ':4096:8'` before those commands.
Wrap commands in `scripts/run_logged.py --label NAME -- ...` to retain their full
output. Training already writes per-evaluation metrics and environment identity.
Runtime and cloud availability must be measured; no quota or cost is promised.

### Kaggle

Import `kaggle_gnn_nca_pilot.ipynb`, select an available GPU and enable Internet
for the corpus download. It embeds the runner, scientific tests, configurations,
license and essential documentation. It uses one GPU and runs the historical
pilot. Save a version with outputs and download the generated result archive to
retain checkpoints. Check the notebook's stated protocol before spending compute.

Regenerate embedded sources with `python build_notebook.py`; check parity with
`python build_notebook.py --check`. Generation also makes a local starter ZIP,
which is excluded from source publication. Notebook GPU execution was not rerun
as part of preparing this repository.

## Model, controls and measurements

The default causal byte LM has **1,857,024 parameters**, four blocks, width 192,
four attention heads and context 128. All stages share 274 output symbols:
256 bytes, 16 binary patch symbols and two frame markers. Reported perplexity is
byte perplexity, not BPE-token or word perplexity.

| Arm | First 100 updates | Next 900 updates | Token embeddings at boundary |
| --- | --- | --- | --- |
| `scratch` | Target text | Target text | Preserved |
| `reserve_text` | Disjoint same-domain text reserve | Target text | Preserved |
| `grid_nca` | Periodic grid trajectories | Target text | Restored to initial values |
| `graph_nca` | Rewired graph trajectories | Target text | Restored to initial values |
| `shuffled_graph` | Frequency-preserving shuffled trajectories | Target text | Restored to initial values |

Every arm restarts its optimizer/schedule at the boundary. Each seed sees
4,096,000 total training tokens per arm; three paired seeds yield 15 runs.
The graph keeps eight typed incoming/outgoing neighbors per node, with no self
or duplicate neighbors. The grid uses the same message-passing implementation;
this is a topology study, not a sparse-kernel versus convolution speed benchmark.
One cached graph/data seed and fixed spatial patch ordering limit generalization.
Random rules are not filtered for useful dynamics or tested on held-out rules.

The default corpus has disjoint byte ranges: 20% text reserve, 60% target train,
10% validation and 10% test. The declared target is 2.5 validation nats/byte.
First recorded crossings are used without interpolation; unreached comparisons
remain unresolved. Reports include target-text tokens, total training tokens,
training seconds and defined pipeline seconds. Pipeline timing includes synthetic
preparation, text preprocessing, setup, training and validation, but excludes
network/package startup, cloud queues and final checkpoint writing. See the
[historical protocol](docs/PILOT_PROTOCOL.md) for exact accounting and limitations.

The next controlled protocol separates K synthetic updates from N common
language updates, skips synthetic training for scratch and matches language data
order, head initialization, optimizer and schedule. Held-out-rule synthetic
learning and boundary checkpoints must be established before a confirmatory
claim. Previously inspected test data must not be reused for selecting that claim.

## Evidence and research records

- [Validation record](VALIDATION.md): historical CPU diagnostic and current check scope.
- [Compact historical evidence](docs/results/README.md): original metrics, summaries,
  verification records, retrospective plots and SHA-256 provenance.
- [Logbook](LOGBOOK.md): changes, tests, failures, observations and inferences.
- [Check transcripts](logs/checks.jsonl): append-only machine records of commands/results.
- [Changelog](CHANGELOG.md): contributor-facing changes.

Retain full `environment.json`, data manifest, metrics, `summary.csv`, `report.json`,
plots, checkpoints and `verification.json` for actual experiments. Do not commit
raw corpora, token arrays or checkpoints. The complete historical archive remains
local; no public download location has yet been established. Compact published
metrics permit recalculation but cannot replace checkpoint/data verification.

## Repository layout and contributing

```text
experiment.py                  Historical generation/training/reporting runner
verify.py, plot_results.py     Checkpoint verification and figure export
configs/, tests/              Declared budgets and scientific/regression checks
scripts/                      Logged checks, offline smoke, publication maintenance
build_notebook.py              Standalone Kaggle notebook generator
sources/                      Pinned upstream MIT reference, not runtime code
docs/results/                 Compact evidence and provenance
.github/                      CPU CI and contribution templates
```

Historical helper scripts accept `--help`; `audit_kaggle_output.py` and
`diagnose_prepretraining.py` require the original full artifacts. `write_validation.py`
exports a new diagnostic report without replacing historical validation by default.
`fetch_reference.py` defaults to the pinned revision; do not overwrite reference
provenance as part of ordinary checks.

See [CONTRIBUTING.md](CONTRIBUTING.md). Reproducibility fixes, held-out-rule
learning diagnostics, faithful grid controls and the corrected stage protocol
are useful starting points. Keep negative results and failed tests in the logbook.

In the original research workspace, edit root sources and run:

```bash
python build_notebook.py
python scripts/sync_prod.py
python scripts/sync_prod.py --check
```

`Prod/` is the selected standalone repository. Publish its contents, including
hidden `.github`, `.gitignore` and `.gitattributes`, rather than the parent
workspace. `PUBLICATION_MANIFEST.json` inventories every selected file. In a
standalone checkout, use `python scripts/check_release.py --refresh` after
changes/tests, then `python scripts/check_release.py`. New publishable files
must be selected in `publication.json`. Ignore rules alone do not select a release.

## License and references

Original project code and documentation: **Apache License 2.0**, see
[LICENSE](LICENSE) and [NOTICE](NOTICE). The pinned upstream snapshot remains
MIT-licensed; see [third-party notices](THIRD_PARTY_NOTICES.md).

Scientific reference: [Training Language Models via Neural Cellular Automata, v1](https://arxiv.org/html/2603.10055v1),
with [author code](https://github.com/danihyunlee/nca-pre-pretraining).
This project is an independent small-scale graph-topology study.


In [ ]:
%%writefile LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.


In [ ]:
%%writefile NOTICE
GNNs in NCA — graph cellular automata pre-pre-training research
Copyright 2026 GNNs in NCA contributors

Original project code and documentation are licensed under Apache License 2.0.
See LICENSE. This notice does not modify any license terms.

The pinned source under sources/nca-pre-pretraining-reference/ is
Copyright (c) 2026 Dan Lee and remains under its included MIT license.
It is reference material, not a runtime dependency or an assertion of endorsement.

Tiny Shakespeare is downloaded or supplied at runtime, not included here.
Dependencies are distributed separately under their respective licenses.


In [ ]:
%%writefile THIRD_PARTY_NOTICES.md
# Third-party material

The unchanged snapshot in `sources/nca-pre-pretraining-reference/` comes from
[danihyunlee/nca-pre-pretraining](https://github.com/danihyunlee/nca-pre-pretraining),
revision `bdd1c71e04eb2e5a2c91481faf278e95846b4431` (12 March 2026).
Copyright (c) 2026 Dan Lee. It remains under the included
[MIT license](sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/LICENSE).
`reference.json` records URLs/hashes. It is not imported by the pilot; its extra
dependencies are not required for this project's checks.

The methodology audit discusses
[Training Language Models via Neural Cellular Automata, v1](https://arxiv.org/html/2603.10055v1).
The paper and competition rules PDF are not redistributed here.

The default corpus is fetched from the
[Tiny Shakespeare source](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt)
at runtime. A data manifest records its SHA-256 and byte splits. Review the source's
terms before redistributing derived data bundles. Raw corpus bytes, cached arrays
and model checkpoints are excluded from this source snapshot.

NumPy, PyTorch and Matplotlib are installed separately under their own licenses.
Version 2 additionally uses separately installed tiktoken and PyArrow. Its NCA
port is based on the retained MIT reference, with the explicit changes recorded
in `docs/TRANSFER_V2_PROTOCOL.md`; the standalone notebook includes that MIT
license. WikiText-2 is downloaded at runtime from the pinned Salesforce dataset
revision and retains its CC-BY-SA-3.0/GFDL terms. Dataset/checkpoint artifacts
remain excluded from the public source selection.
`docs/results/` contains project measurements and plots; its provenance manifest
distinguishes copied originals from adapted summaries.


In [ ]:
%%writefile CONTRIBUTING.md
# Contributing

Contributions to reproducibility, controls, diagnostics and documentation are
welcome. Read [the research context](RESEARCH_CONTEXT.md) and
[methodology audit](METHODOLOGY_AUDIT.md) first. `experiment.py` preserves pilot 1;
`transfer_v2/` implements the corrected independent-stage transfer experiment.
Read [the v2 protocol](docs/TRANSFER_V2_PROTOCOL.md) before changing its controls.

## Setup and checks

Use Python 3.12 and a virtual environment; install `requirements.txt`. The README
gives Windows and Unix commands. `requirements-tested.txt` records the direct
versions checked here. The broader dependency ranges are not a claim that every
combination has been tested. Install an appropriate CUDA PyTorch build separately
for GPU work. CPU checks need no cloud account.

Run from the repository root:

```bash
python scripts/run_logged.py --label scientific-tests -- python -m unittest discover -s tests -v
python scripts/run_logged.py --label offline-smoke -- python scripts/smoke_check.py
python scripts/run_logged.py --label notebook-parity -- python build_notebook.py --check
python scripts/run_logged.py --label v2-offline -- python scripts/smoke_transfer_v2.py
python scripts/run_logged.py --label v2-notebook -- python build_transfer_notebook.py --check
```

The offline smoke check creates an isolated temporary run using generated text,
trains all five arms, reloads checkpoints and exports plots. It does not download
data or measure a transfer benefit. Do not use Python `-O`: historical scientific
checks use assertions.

## Propose changes

Open an issue, then submit a focused pull request with:

- The problem, resulting behavior and any changed scientific assumptions.
- Commands, hardware, outputs, failures and remaining limitations.
- An entry in `LOGBOOK.md` and `CHANGELOG.md` under `Unreleased`.
- Regression coverage for changed controls, integrity checks or measurements.

Declare budgets, embedding/optimizer policy, data/rule/topology seeds and metrics
before confirmatory runs. Report language-stage and cumulative costs separately.
Use fresh data/run directories, retain failed runs and document why they failed.
Never tune a protocol to improve an already inspected test result. Submit small
metrics and provenance; link large artifacts in an issue or release with SHA-256
checksums rather than committing corpora, checkpoints or ZIPs.

## Publication maintenance

In the original research workspace, edit root files, regenerate both notebooks
with `build_notebook.py` and `build_transfer_notebook.py`,
then run `python scripts/sync_prod.py`. `Prod/` is the standalone repository to
publish. `python scripts/sync_prod.py --check` detects drift without writing.
Finish with these sync commands directly after logged checks; wrapping a sync
in the logger appends another log entry after the copy and therefore creates drift.

In a standalone checkout, run `python scripts/check_release.py --refresh` after
changes and tests, then `python scripts/check_release.py`. The inventory includes
only distributable files. Do not publish the parent research workspace.

For release preparation, `python scripts/git_roundtrip_check.py` also checks a
temporary Git staging/checkout cycle, preserving historical byte checksums across
line-ending settings. In the research workspace pass `--root Prod`. It creates
no commit or remote and removes its temporary repository afterwards.

By submitting material for inclusion, you contribute it under Apache-2.0 unless
explicitly stated otherwise. Only submit material you may share. Preserve
third-party notices and provenance. Discuss results respectfully, include
negative evidence and make reproduction possible for other contributors.


In [ ]:
%%writefile RESEARCH_CONTEXT.md
# Research context: GNNs in NCA pre-pre-training

Updated: 17 October 2026.

The research question is whether **graph-NCA data improves the transformer
initialization acquired during pre-pre-training**, before ordinary language
pre-training. Grid NCA and graph NCA are synthetic sources for that first stage.
Neither is a replacement for the subsequent language-model training stage.

Primary references to retain:

- Paper, version 1: https://arxiv.org/html/2603.10055v1
- Author repository: https://github.com/danihyunlee/nca-pre-pretraining/tree/main
- Pinned source revision inspected: `bdd1c71e04eb2e5a2c91481faf278e95846b4431`
  (12 March 2026).
- Local source snapshot and checksums:
  `sources/nca-pre-pretraining-reference/reference.json`, with files beneath
  `sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/`.

The correct primary comparison is:

| Initialization entering language pre-training | Synthetic pre-pre-training | Language pre-training |
| --- | --- | --- |
| Random / no pre-pre-training control | None | Same N text updates |
| Grid-NCA initialized transformer | K grid-NCA updates | Same N text updates |
| Graph-NCA initialized transformer | K graph-NCA updates | Same N text updates |
| Shuffled-NCA initialized transformer | K shuffled-data updates | Same N text updates |

Use identical language data order, optimizer initialization, learning-rate
schedule, batch/context, evaluation and stage length. Compare graph vs grid as
pre-pre-training treatments. A text-only model remains a valid **no
pre-pre-training control**, as in the paper; it must not get an unacknowledged
head start on the common language-stage clock.

Report transfer efficiency on language-stage tokens/time first. Separately
report cumulative synthetic + language cost and generation cost. The paper also
reports a combined token-cost metric in Appendix D; these questions must remain
distinct rather than removing cost accounting altogether.

Pilot 1 is preserved locally in `Kaggle Outputs/gnn_nca_results_1.zip`; compact
public evidence is in `docs/results/pilot1/`. All 15 runs and
checkpoints passed independent verification. Its design was a **matched total
budget replacement pilot**: scratch received 1000 text updates; synthetic arms
received 100 synthetic + 900 text. Its negative result is valid for that design
but is not a clean replication of the paper's transfer experiment. Its scratch
scheduler also restarted after 100 text updates, unlike the synthetic arms'
language-stage schedule.

See `METHODOLOGY_AUDIT.md` for the specific mismatches. Existing aligned logs show
no sustained graph benefit, so correcting the framing must not be described as
discovering a positive result. At 900 target-text updates, mean validation NLLs
are 1.7542 (scratch), 1.8002 (grid), and 1.8170 (graph), with the scratch schedule
still confounded. No matched-text final scratch test checkpoint was saved.

Do not overwrite original run files or silently relabel the old experiment.
Establish useful held-out-rule NCA learning, reproduce grid-NCA transfer at an
affordable scale, then introduce the graph modification. Avoid interpreting a
one-time generation timing difference as an optimized graph-kernel speedup.

## Implemented next run

The independent-stage implementation is now `transfer_v2/`, configured by
`configs/transfer_v2.json`, with notebook `kaggle_nca_transfer_v2.ipynb`.
See `docs/TRANSFER_V2_PROTOCOL.md`. It restores categorical NCA/rule filtering/
burn-in and uses a small Llama-style model, independent 1000-update PPT and
2000-update language budgets, grid/shuffle/no-PPT controls, stage-boundary
checkpoints and a held-out-rule prediction gate. Language heads, data order,
optimizers and schedules match across arms. The language confirmation split is
reserved by default. The returned v2 GPU development run is now measured; see
[its results review](docs/results/transfer_v2_run1/REVIEW.md). All three grid seeds
passed the synthetic gate and reached their paired scratch endpoint after
1600–1800 language updates versus 2000, with a geometric language-token speedup
of 1.178x. Shuffled NCA achieved the same aggregate token speedup. Mean final
validation NLLs were 5.266091 (scratch), 5.233321 (grid), and 5.231528 (shuffled).
This supports a small synthetic-initialization transfer effect in this setup;
it does not isolate local dynamics or establish a graph benefit. The recorded
cumulative cost of grid NCA was about 13% higher at the reference loss.
The official test remains unused. Graph NCA has not been run under v2.

## Next stage: graph transfer v3

`transfer_v3/`, `configs/transfer_v3.json` and
`kaggle_nca_transfer_v3.ipynb` implement the next experiment; see
[the v3 protocol](docs/TRANSFER_V3_PROTOCOL.md). It compares no-PPT, grid, graph,
shuffled grid and shuffled graph before the same 2000-update language stage.
All synthetic arms receive 1000 PPT updates. Each paired seed gets an independent
rule cache/topology. Grid and graph share frozen rule weights and paired random
draws; selection requires both profiles to pass the same complexity band.
The graph changes donor connectivity while preserving degrees/ordered slots.

The primary comparison is graph-PPT versus grid-PPT; graph versus its shuffled
control is needed to assess ordered-trajectory structure. Comparisons to no-PPT
and cumulative cost remain additional results. Actual initial core/head weights
are saved, so cross-platform checks can verify those tensors directly. Both grid
and graph must pass the synthetic gate before any language run. This is new
software with CPU checks; no v3 GPU acceleration result exists yet. Preserve
the measured v2 result separately and leave official-test confirmation unused.


In [ ]:
%%writefile METHODOLOGY_AUDIT.md
# Where pilot 1 diverges from NCA pre-pre-training

This audit uses the [paper v1](https://arxiv.org/html/2603.10055v1), the
[author repository](https://github.com/danihyunlee/nca-pre-pretraining/tree/main)
at revision `bdd1c71e04eb2e5a2c91481faf278e95846b4431`, our `experiment.py`, and
the independently verified first Kaggle output. The references and source
checksums are retained locally in `sources/nca-pre-pretraining-reference/`.

## Main correction

Our code really did train synthetic-then-text; it did not omit the language
stage. The error was in the experimental question and in interpreting its result.
The pilot fixed the total number of updates and replaced some text training with
synthetic training, rather than first measuring how a pre-pre-trained
initialization changes an otherwise identical language-training run.

That is a resource-allocation experiment. The intended first experiment
is a **transfer-initialization experiment**, followed by an additional cost audit.
The earlier roughly 43% timing penalty is a correct calculation for pilot 1's
declared total-cost metric. It is not a clean measurement of the paper's
language-stage convergence question, and should not be used as a decisive
rejection of pre-pre-training.

The paper does retain a no-pre-pre-training scratch control. Keeping that control
is appropriate; giving it extra target-language exposure at the nominal language
stage boundary is the problem. Graph and grid NCA should be compared as
alternative pre-pre-training treatments feeding the same downstream procedure.
The paper's Appendix D additionally counts synthetic and language tokens
together; total cost remains a useful secondary question.

## Definite design mismatches

### 1. Unequal language exposure and an incorrect scratch boundary

In `experiment.py::run_one`, every arm runs `pre_steps=100`, then
`total_steps-pre_steps=900` updates. For `scratch`, both stages use target text.
For grid/graph/shuffle, the first stage uses synthetic data. Therefore:

| Arm | Updates before nominal common language stage | Nominal common stage | Target-text updates in total |
| --- | --- | --- | ---: |
| Scratch | 100 target-text updates | 900 target-text updates | 1000 |
| Grid / graph / shuffled NCA | 100 synthetic updates | 900 target-text updates | 900 |

Scratch already has learned byte embeddings and a text-trained backbone at the
nominal language-stage origin. Its first 100 text updates also use a separate
short cosine cycle, followed by an optimizer/scheduler restart. Replotting against
target-text tokens removes the exposure mismatch on the x-axis, but cannot undo
the different scheduler or recover the missing final scratch test checkpoint.

For the next controlled run, use separate `ppt_steps=K` and `language_steps=N`.
Skip the PPT loop entirely for scratch, and start its language optimizer at
language step zero alongside the transferred initializations.

### 2. We changed the generator's function family

The reference [generator](https://github.com/danihyunlee/nca-pre-pretraining/blob/bdd1c71e04eb2e5a2c91481faf278e95846b4431/utils/nca.py)
uses categorical one-hot states, a spatial perception layer with four channels,
then a pointwise hidden layer with 16 channels and categorical output. Our pilot
uses signed binary inputs and a different 9-to-16-to-2 MLP. Thus it is not just a
smaller grid or a different implementation of the same reference rule family.

Our test proving convolution/message-passing equivalence is correct for **our
own binary rule**. It does not establish equivalence to the author's NCA.

The default reference state alphabet has 10 values, yielding 10,000 possible
2x2 patch patterns. Our binary alphabet yields 16. The reference launcher actually
allocates 64,000 model vocabulary slots; possible patch patterns and model output
vocabulary must not be conflated. Our shared 274-way byte/synthetic vocabulary is
a separate design choice.

### 3. Complexity selection and burn-in were omitted

The reference [PPT launcher](https://github.com/danihyunlee/nca-pre-pretraining/blob/bdd1c71e04eb2e5a2c91481faf278e95846b4431/scripts/prepretraining/nca_prepretraining.sh)
enables rule selection and skips 10 initial automaton steps. Our generator
accepts every sampled rule and includes its initial random frame.

From the returned arrays, **36.33% of graph trajectories** and **35.74% of grid
trajectories** have identical final two frames. For a deterministic rule this
means that rollout has reached a fixed point. This is direct evidence that the
unfiltered corpus contains trivial late dynamics. It does **not** establish that
this alone caused negative transfer.

Our logged gzip ratios are not comparable to the reference filter threshold.
We compressed uint16 token IDs with frame markers; the author filter removes
markers and compresses its token representation separately per trajectory. Raw
integer width, alphabet, sequence length and burn-in all affect the ratio. The
new diagnostic explicitly labels a marker-free int32 calculation as
non-equivalent because our cached rollouts still lack reference burn-in.

Do not simply apply the author's 0.5 threshold to our binary uint16 files and
call that a faithful filter. First standardize generation and serialization.

### 4. We never established that the PPT stage learned the intended task

The [author PPT trainer](https://github.com/danihyunlee/nca-pre-pretraining/blob/bdd1c71e04eb2e5a2c91481faf278e95846b4431/src/nca_ppt.py)
constructs separate training and held-out rule sets. It evaluates synthetic
prediction loss and has facilities for next-grid generalization evaluation.
Its dataset masks the initial context grid from prediction loss.

Our runner only logs **text validation loss**, even while the model is training
on synthetic symbols. It does not log synthetic training/validation loss, does
not hold out NCA rules, and does not save the PPT checkpoint before transferring.
We can verify that the updates executed, but cannot establish from the artifacts
whether the model learned transferable rule inference, merely frequencies, or
trivial copying. The first-stage checkpoint cannot be recovered from the final
language checkpoint.

Random cropped windows, unmasked context and repeated small cached trajectories
also differ from the reference trajectory-aligned task. Frame markers account
for 11.11% of our synthetic tokens. Predicting their periodicity is an easy task;
a falling overall NCA loss would therefore need patch-only and held-out-rule
checks before being treated as mechanistic evidence.

### 5. Both optimization stages use an untested common configuration

The reference PPT code uses Adam with learning rate 1e-4 and no weight decay in
its main setup. The language launcher uses a separate learning rate/optimizer
configuration. Our pilot uses AdamW at 6e-4, weight decay 0.1, and gradient clipping
in both stages. Those changes are not automatically wrong for a tiny model, but
we never validated the PPT learning curve before committing to them.

The paper's language-stage weight decay is 1e-4, versus our 0.1. The much larger
value and other hyperparameter differences are plausible sensitivity issues;
they are not proven causes of the result. Raw absolute PPT token counts also do
not establish undertraining by themselves because model size changed greatly.

### 6. Model, context and text task were changed simultaneously

Our 1.86M-parameter model uses learned absolute position embeddings and a GELU/
LayerNorm transformer with context 128 and byte-level Shakespeare training.
The reference [model implementation](https://github.com/danihyunlee/nca-pre-pretraining/blob/bdd1c71e04eb2e5a2c91481faf278e95846b4431/utils/models.py)
uses a Llama backbone with rotary positions; its principal launcher uses context
1024. The paper's reported model scales and language datasets are far larger.

Small-scale experiments remain useful, but reducing the state alphabet, context,
model family, corpus and pretraining budget together does not preserve the
conditions under which transfer was demonstrated. Whether our tiny model can
learn the relevant in-context dynamics is a question to measure, not assume.

### 7. The graph extension was introduced before validating grid transfer

We retained spatial 2x2 patch serialization while rewiring graph edges. Many
serialized spatial neighbors then cease to be actual graph neighbors, while
actual graph neighbors can be far apart in the sequence. This may make rule
inference harder or change complexity. It is an additional hypothesis, not a
guaranteed improvement from using a GNN.

First implement the reference grid update as relational message passing on the
same lattice and verify equivalent outputs. Changing connectivity should then
be a separately declared treatment. The first pilot's grid-vs-graph comparison
still is a valid comparison of its two synthetic treatments, but neither has
established the intended grid-transfer effect.

## What we should retain

- Causal next-token prediction on serialized trajectories is appropriate.
- Frozen randomly sampled NCA rules are appropriate; the generator does not need
  a separate learned task before being used as synthetic data.
- **Reinitializing input/output token embeddings is part of the reference
  transfer protocol.** It is not an error to fix by preserving arbitrary NCA
  symbol semantics. Transfer the backbone; initialize a common byte/text head
  consistently across arms. Reference code:
  [language transfer](https://github.com/danihyunlee/nca-pre-pretraining/blob/bdd1c71e04eb2e5a2c91481faf278e95846b4431/src/openwebtext_pt.py).
- Paired seeds, fixed language evaluation, checksum verification, a grid control
  and a frequency-preserving shuffled control remain useful.
- Total-cost accounting remains useful, but must be presented alongside the
  primary language-stage transfer measurements rather than replacing them.

Our separate-text arm preserved its learned byte embeddings, whereas the primary
synthetic arms reset theirs. That is a meaningful practical text baseline, but
does not isolate backbone transfer under a common embedding policy. If retaining
a text-PPT baseline in the next run, distinguish text-PPT with a reset language
head from a separate text-PPT embedding-preservation ablation. Do not silently
compare different transfer policies and attribute the entire difference to NCA
structure.

## What existing logs say after correcting the exposure axis

The retrospective plot is [the retained retrospective plot](docs/results/pilot1/language_aligned_curves.png).
The exact calculations and generator checks are saved beside it in
`docs/results/pilot1/prepretraining_diagnosis.json`.
Plot lines are means and shaded bands are the minimum/maximum across three
seeds, not confidence intervals.

Mean validation loss across the original three seeds:

| Target-text updates seen | Scratch | Grid PPT → text | Graph PPT → text | Shuffled PPT → text |
| ---: | ---: | ---: | ---: | ---: |
| 100 | 2.6141 | 2.6100 | 2.6428 | 2.6184 |
| 200 | 2.4512 | 2.4344 | 2.4460 | 2.4398 |
| 500 | 2.0193 | 2.0065 | 2.0287 | 2.0133 |
| 900 | 1.7542 | 1.8002 | 1.8170 | 1.8099 |

There are small intermediate improvements on this axis, but no sustained graph
advantage. Scratch's scheduler remains different, so this is diagnosis rather
than a clean replacement experiment. These are **validation** comparisons;
scratch was not saved at the matched 900-text-update point for test evaluation.
Correcting the question does not turn the old result into proof of success.

## Correct next protocol

1. **Separate stage budgets.** Define PPT length K and language length N
   independently. Grid, graph and shuffled arms run K+N updates. The no-PPT arm
   runs N text updates from a fresh initialization, not K+N text updates under a
   different schedule. If a fixed-total-cost arm is desired, label it separately.
2. **Validate synthetic learning before transfer.** Hold out entire rules;
   retain full trajectory boundaries; mask initial context; log synthetic loss,
   patch-only loss/accuracy and held-out-rule prediction against frequency/copy
   controls. Save the PPT checkpoint and rule/data identities.
3. **Recover a faithful grid reference first.** Preserve its categorical state
   representation, update structure, patch encoding, complexity selection and
   burn-in. Scale the transformer and budgets transparently rather than changing
   all mechanistic assumptions at once. Audit reference-code caveats below.
4. **Enter an identical language stage.** Use common language embedding/head
   initialization, architecture, data sequence, optimizer state, scheduler and N
   across no-PPT, grid, graph and shuffled initializations. Full-backbone transfer
   is the primary setup; attention-only transfer can be a declared ablation.
5. **Report both questions.** Plot loss vs language tokens/time from stage zero,
   and separately loss vs total training cost including PPT/generation. The
   paper's convergence comparison uses the scratch endpoint as a reference;
   our arbitrary 2.5 threshold is an additional metric, not an exact reproduction.
6. **Add the graph treatment only after the grid check.** Match rule family,
   tokenization and data-complexity policy. Separate changing the numerical
   implementation from changing the graph topology.

Use development validation for generator/hyperparameter decisions, and reserve
untouched confirmation data for the next claims. Pilot 1's test set has already
been examined. No new GPU benchmark has been launched by this audit.

## Reference-code caveats to record rather than silently inherit

The public code is useful but should not be treated as a bit-exact executable
specification of every sentence in the paper:

- Paper v1 states temperature 1e-3; the pinned launcher passes 1e-4. Record the
  chosen value explicitly.
- `src/nca_ppt.py` contains `epoch+1 % args.generate_rules == 0`. Python parses
  that as `epoch + (1 % args.generate_rules) == 0`, not
  `(epoch + 1) % args.generate_rules == 0`. With the launcher value one it only
  triggers at epoch zero, despite the intended periodic refresh. This is a
  static code finding; the author code has not been executed locally.
- `utils/nca.py` applies scalar-width `jnp.pad` to a three-axis array; that pads
  channels as well as spatial dimensions. A port must explicitly decide whether
  to reproduce that behavior or implement the spatial padding described by the
  paper. Do not call the two bit-identical without testing.
- The tokenizer constructs masked marker targets, but `NCADataset.__getitem__`
  reconstructs targets from `seq`, then masks the initial context grid. Therefore
  do not claim the currently executed reference loss necessarily masks all
  markers just because the tokenizer produces a masked target array.
- Model constructors have weight-tying defaults that need explicit inspection;
  the paper's description alone is insufficient to infer launcher behavior.

These source caveats are separate from the faults in our independent pilot; they
do not excuse its unmeasured synthetic learning or unequal language budgets.


In [ ]:
%%writefile VALIDATION.md
# Local verification — 17 October 2026

This records the historical CPU diagnostic. The subsequently returned GPU pilot
is documented in [the audited review](docs/results/pilot1/REVIEW.md); no new GPU
benchmark was run during publication preparation. New check transcripts are in
`logs/checks.jsonl`, with scope and results in `LOGBOOK.md`.

- Scientific checks: **11 passed** (`python -m unittest discover -s tests -v`).
- Full model: **1,857,024 parameters**, four layers, width 192, context 128.
- Diagnostic budget: 60 updates per arm, batch 4, 30,720 total training tokens per arm, one paired seed (17).
- All five checkpoints reloaded; held-out test losses reproduced with **zero absolute error**.
- Initial parameter checksums match across all five arms.
- The 2.5 validation-nats/byte target was not reached. All time-to-target comparisons remain unresolved.

| Arm | Final validation NLL | Held-out test NLL |
| --- | ---: | ---: |
| scratch | 2.798420 | 2.889472 |
| reserve_text | 2.806562 | 2.896473 |
| grid_nca | 3.196839 | 3.298223 |
| graph_nca | 3.175359 | 3.277369 |
| shuffled_graph | 3.185969 | 3.284273 |

The short check favors text-only training over graph-NCA on language loss.
Graph NCA slightly improves loss relative to grid NCA and its shuffled control in this one-seed diagnostic,
but this is insufficient to claim a topology benefit, statistical significance, or GPU acceleration.
The Kaggle pilot uses the predeclared longer budget; do not treat this CPU diagnostic as that pilot.

## Historical evidence (original paths in the research workspace)

- `runs/cpu_verified/verification.json`: independently recomputed test losses and valid checkpoint checksums.
- `runs/cpu_verified/summary.csv`: metrics, parameter counts, initialization checksums, and checkpoint hashes.
- `runs/cpu_verified/report.json`: explicitly censored time-to-target comparisons.
- `runs/cpu_verified/curves.svg`: validation-loss traces with synthetic steps counted.
- `runs/cpu_verified/environment.json`: exact runtime and source identity.
- `data/cpu_check/manifest.json`: text ranges, checksums, graph identity, and generator diagnostics.

Experiment source SHA-256: `76b535acbf08050295648d5692d46690366c1f4802d5668c99edeca003cf1981`.

The separate six-update smoke run also completed and reloaded all five checkpoints successfully.

Compact publication copies of summary, report and checkpoint verification are
in `docs/results/cpu_check/`. Full datasets/checkpoints remain local. The source
hash recorded here identifies the historical experiment, not the entire evolving repository.

## Version 2 checks — 17 October 2026

These are new CPU/software/data-path checks, separate from the historical GPU
pilot above. They do not establish learning acceleration.

- All **33** unit/scientific/integrity checks passed, including **16** v2 tests.
- Offline v2 workflows completed for normal execution, a deliberately failed
  synthetic gate, and a predeclared confirmation-fixture path. Selected stage
  checkpoints reloaded with zero evaluation error; language head, optimizer,
  schedule and batch-order controls passed. A stopped gate produced no language
  checkpoints.
- Default-sized language model (12,306,048 parameters) accepted the real 50,257
  BPE vocabulary and completed a short forward/backward shape check. This is not
  a full-context GPU memory/performance measurement.
- Real pinned WikiText train/validation preparation passed: 2,403,644 training
  tokens and 248,461 validation tokens. The confirmation test was not downloaded.
- A CPU categorical-rule calibration retained 16 rules from its first 64 screened
  candidates within the declared band; its sampled fresh rollouts had no final
  fixed transition. This is a small generator check, not a corpus-wide guarantee.
- Historical `experiment.py` retains its original SHA-256.

Full commands, outputs, failed attempts and v2 source identities are appended in
`logs/checks.jsonl`. An initial OS-temporary-directory permission failure was
resolved by retaining offline-check artifacts under the workspace's ignored
`tmp/` directory. These checks preceded the returned v2 GPU archive below.

## Returned version 2 GPU evidence — 17 October 2026

The user ran the notebook on a Kaggle Tesla T4 and supplied
`nca_transfer_v2_results_20261017_065212_924967.zip`. This is imported GPU
measurement, distinct from the earlier CPU fixtures and from configured CI.
See [the results review](docs/results/transfer_v2_run1/REVIEW.md) for the
independent audit, raw curves and scope of verification.

All three grid-NCA seeds passed the synthetic gate. All nine language runs used
2000 updates / 8,192,000 input tokens each. Grid and shuffled pre-pre-training
both yielded a 1.178x geometric language-token speedup to the paired scratch
endpoint. Their final mean NLLs were 5.233321 and 5.231528 versus 5.266091 for
scratch. Grid's cumulative-time speedup was 0.886x, so the measured language
savings did not repay pre-pre-training at this horizon. No graph treatment or
official-test evaluation was performed.

Independent local CPU replay loaded all **15 development checkpoints**, with
maximum validation-metric error **1.9563e-7**. Archive CRCs, trusted source and
data identities, actual rule disjointness, shuffled histograms/markers, fitted
controls and raw crossing calculations passed. Actual PPT-transfer and batch
order hashes match. Initial/head/schedule hashes agree within the GPU run, but
are not all recreated bitwise by the newer Windows runtime; the retained
step-zero smoke core differs numerically by at most 7.45e-9. Failed regeneration
attempts and the corrected audit scope remain in the append-only check log.

## Graph-transfer v3 software checks — 17 October 2026

This is new CPU software evidence, separate from the returned v2 GPU result.
All 58 unit/scientific/integrity checks passed, including 25 v3 checks. They
cover degree-preserving graph rewiring, exact zero-rewiring grid recovery,
node-relabeling equivariance of the message update, paired random draws, joint
rule selection, independent caches, graph-only gate rejection and censored
graph-vs-grid reporting, alongside the stage controls retained from v2.

Offline complete, gate-stopped and explicit confirmation-fixture workflows
passed. The complete path uses two paired replicates, all five language arms
and 18 selected-stage checkpoint replays. Stored initial tensors are checked;
an intentionally corrupted initialization is rejected before model replay.
Actual rule-weight disjointness and shuffled histograms/markers are verified.
The fixture confirmation path does not download or inspect the official test.
Full commands and source identities remain in `logs/checks.jsonl`. CI changes
are configured only. No v3 GPU training has run.


## Verify the scientific controls
These checks test causality, initialization, graph degree, convolution equivalence,
patch tokenization, shuffling, trajectory boundaries, and censored target handling.


In [ ]:
def run(*args):
    subprocess.run([sys.executable, *args], check=True)

run('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## GPU smoke run
Six updates per arm exercise CUDA, caching, stage transfer, reporting, and saved
checkpoint verification. Smoke losses are execution checks, not research results.


In [ ]:
SMOKE_DATA = f'data/smoke_{RUN_ID}'
SMOKE_OUT = f'runs/smoke_{RUN_ID}'
run('experiment.py', 'all', '--config', 'configs/smoke.json',
    '--data', SMOKE_DATA, '--out', SMOKE_OUT, '--device', 'cuda')
run('verify.py', '--data', SMOKE_DATA, '--out', SMOKE_OUT, '--device', 'cuda')


## Three-seed pilot
Configuration and target are fixed before looking at pilot outcomes. All arm
comparisons use the same fixed validation and held-out test subsets. There is no
generator selection against test loss. Complete preparation is charged per run
even though data are cached across seeds. See README.md for exact timing scope.
The run uses a fresh output folder; failed runs never overwrite old checkpoints.


In [ ]:
PILOT_DATA = f'data/pilot_{RUN_ID}'
PILOT_OUT = f'runs/pilot_{RUN_ID}'
run('experiment.py', 'all', '--config', 'configs/pilot.json',
    '--data', PILOT_DATA, '--out', PILOT_OUT,
    '--text', str(Path(SMOKE_DATA) / 'input.txt'), '--device', 'cuda')
run('verify.py', '--data', PILOT_DATA, '--out', PILOT_OUT, '--device', 'cuda')
run('plot_results.py', '--out', PILOT_OUT)


## Inspect the result
A pipeline speedup above one favors graph NCA. Report censored seeds and compare
against scratch, grid NCA, and shuffled trajectories. Three seeds are exploratory;
they cannot prove a general benefit. Byte perplexity is not comparable to BPE
perplexity. This pilot does not test language reasoning or train a GNN rule.


In [ ]:
import json
import csv
from IPython.display import SVG, display

with (Path(PILOT_OUT) / 'summary.csv').open(encoding='utf-8') as handle:
    display(list(csv.DictReader(handle)))
display(SVG(filename=str(Path(PILOT_OUT) / 'curves.svg')))
result = json.loads((Path(PILOT_OUT) / 'report.json').read_text())
display(result['comparisons'])
display(result['aggregate'])
print('Never substitute a made-up speedup for an unresolved target.')


## Download the verifiable result bundle
The archive includes the exact input tokens and checkpoints, so held-out losses
can be independently recomputed. Save the notebook version with outputs too.


In [ ]:
import zipfile
from IPython.display import FileLink

archive = Path('/kaggle/working') / f'gnn_nca_results_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for file in ROOT.rglob('*'):
        if file.is_file() and '__pycache__' not in file.parts:
            z.write(file, file.relative_to(ROOT))
print('Saved:', archive, f'({archive.stat().st_size / 1e6:.1f} MB)')
display(FileLink(str(archive)))
